# Avance 02 — INEGI ATUS — Etapa 2: validación y perfil de calidad
**Equipo A&C | Inteligencia de Negocios**

Trabajamos con los **62,424 registros aptos** obtenidos en la Etapa 1 (2018–2025, 16 alcaldías de CDMX). Revisamos integridad, temporalidad, geografía, duplicados y variables de personas lesionadas/fallecidas. **No eliminamos ni modificamos registros de entrada.** Guardamos evidencias de cada diagnóstico para definir las reglas de la Etapa 3.

ATUS registra accidentes y no debe unirse por fila ni sumarse indiscriminadamente con SSC o C5. Los indicadores de personas se evalúan como conteos, no como tasas de letalidad.

## 1. Recuperar la salida de la Etapa 1
Buscamos la carpeta desde la ubicación actual y sus carpetas superiores. Esta etapa **no necesita volver a leer el ZIP nacional**.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd
import numpy as np
from IPython.display import display

ETAPA1 = None
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidata = base / 'resultados' / 'resultados_INEGI_ATUS' / 'etapa1'
    if (candidata / 'INEGI_ATUS_etapa1.csv').is_file():
        ETAPA1 = candidata
        break
if ETAPA1 is None:
    raise FileNotFoundError('No encontramos resultados/resultados_INEGI_ATUS/etapa1/INEGI_ATUS_etapa1.csv')
ENTRADA = ETAPA1 / 'INEGI_ATUS_etapa1.csv'
SALIDA = ETAPA1.parent / 'etapa2'
SALIDA.mkdir(parents=True, exist_ok=True)
resumen_etapa1 = pd.read_csv(ETAPA1 / 'resumen_delimitacion_INEGI_ATUS.csv', encoding='utf-8-sig')
ESPERADAS = int(resumen_etapa1.loc[0, 'filas_aptas'])
assert ESPERADAS == 62424, f'La etapa 1 reporta {ESPERADAS:,} registros, no 62,424. Revisar la versión.'
CABECERA = pd.read_csv(ENTRADA, nrows=0, encoding='utf-8-sig').columns.tolist()
ORIGINALES = [c for c in CABECERA if c not in ['archivo_origen','fila_origen','alcaldia_normalizada','motivo_exclusion']]
assert len(ORIGINALES) == 46, f'Se esperaban 46 columnas originales, encontramos {len(ORIGINALES)}'
print('Entrada:', ENTRADA)
print('Salida:', SALIDA)
print('Filas esperadas:', ESPERADAS)
print('Columnas originales:', len(ORIGINALES))
print('Columnas de entrada:', len(CABECERA))

Entrada: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa1/INEGI_ATUS_etapa1.csv
Salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa2
Filas esperadas: 62424
Columnas originales: 46
Columnas de entrada: 50


## 2. Contrato de validación
Validamos año, mes, día y hora de forma separada; cuando es posible, construimos una fecha para detectar días imposibles. `ID_HORA=99`, `ID_MINUTO=99` u otros códigos especiales **no se interpretan automáticamente como horas reales**: los marcamos para revisión. Para las personas fallecidas y lesionadas, los valores negativos o no numéricos se marcan como no aptos para sumarse; **no reemplazamos los valores originales**.

No suponemos que dos filas iguales en las 46 columnas representan necesariamente el mismo accidente: la fuente no contiene en estas columnas un folio único que permita decidirlo sin revisión de su unidad de observación.

In [2]:
OBLIGATORIAS = ['ANIO','MES','ID_DIA','ID_HORA','ID_MINUTO','CVE_ENT','CVE_MUN','CVEGEO','alcaldia_normalizada']
PERSONAS = ['CONDMUERTO','CONDHERIDO','PASAMUERTO','PASAHERIDO','PEATMUERTO','PEATHERIDO',
            'CICLMUERTO','CICLHERIDO','OTROMUERTO','OTROHERIDO','NEMUERTO','NEHERIDO']
faltan = [c for c in OBLIGATORIAS + PERSONAS if c not in CABECERA]
if faltan:
    raise KeyError(f'Faltan columnas necesarias: {faltan}')
print('Contrato de columnas: correcto')
print('Variables de personas:', ', '.join(PERSONAS))

Contrato de columnas: correcto
Variables de personas: CONDMUERTO, CONDHERIDO, PASAMUERTO, PASAHERIDO, PEATMUERTO, PEATHERIDO, CICLMUERTO, CICLHERIDO, OTROMUERTO, OTROHERIDO, NEMUERTO, NEHERIDO


## 3. Perfil por bloques y comprobaciones
Procesamos por bloques de 20,000 filas para no cargar todo el CSV simultáneamente. Calculamos duplicados exactos por las 46 columnas originales (sin considerar columnas derivadas) mediante huellas de fila; las coincidencias son **alertas**, no eliminaciones. Guardamos ejemplos limitados de casos para revisión.

In [3]:
TOTAL = 0
conteo_anios = Counter()
conteo_alcaldias = Counter()
alertas = Counter()
faltantes = Counter()
valores_no_vacios = Counter()
valores_distintos = defaultdict(set)
perfil_personas = defaultdict(Counter)
huellas_vistas = set()
filas_duplicadas_adicionales = 0
muestras = defaultdict(list)
LIMITE_MUESTRA = 40

def registrar_muestra(nombre, bloque, mascara):
    disponibles = LIMITE_MUESTRA - len(muestras[nombre])
    if disponibles > 0 and mascara.any():
        columnas = [c for c in ['archivo_origen','fila_origen','ANIO','MES','ID_DIA','CVEGEO','alcaldia_normalizada'] if c in bloque]
        muestras[nombre].extend(bloque.loc[mascara, columnas].head(disponibles).to_dict('records'))

for bloque in pd.read_csv(ENTRADA, dtype=str, encoding='utf-8-sig', chunksize=20000):
    TOTAL += len(bloque)
    conteo_anios.update(bloque['ANIO'].value_counts(dropna=False).to_dict())
    conteo_alcaldias.update(bloque['alcaldia_normalizada'].value_counts(dropna=False).to_dict())
    for c in CABECERA:
        s = bloque[c].fillna('').astype(str).str.strip()
        es_faltante = s.eq('')
        faltantes[c] += int(es_faltante.sum())
        valores_no_vacios[c] += int((~es_faltante).sum())
        # Cardinalidad para columnas de códigos; no guardamos todas las filas ni valores de texto libre.
        if c in ['MES','ID_DIA','ID_HORA','ID_MINUTO','CVE_ENT','CVE_MUN','CVEGEO','TIPACCID','CLASACC','ESTATUS']:
            valores_distintos[c].update(s.unique().tolist())
    anio = pd.to_numeric(bloque['ANIO'], errors='coerce')
    mes = pd.to_numeric(bloque['MES'], errors='coerce')
    dia = pd.to_numeric(bloque['ID_DIA'], errors='coerce')
    hora = pd.to_numeric(bloque['ID_HORA'], errors='coerce')
    minuto = pd.to_numeric(bloque['ID_MINUTO'], errors='coerce')
    fecha = pd.to_datetime(dict(year=anio, month=mes, day=dia), errors='coerce')
    pruebas = {
        'anio_fuera_2018_2025': ~anio.between(2018, 2025),
        'mes_invalido_o_faltante': ~mes.between(1,12),
        'dia_invalido_o_faltante': ~dia.between(1,31),
        'fecha_calendario_invalida': fecha.isna(),
        'hora_fuera_0_23_o_faltante': ~hora.between(0,23),
        'minuto_fuera_0_59_o_faltante': ~minuto.between(0,59),
        'entidad_distinta_09': bloque['CVE_ENT'].fillna('').str.strip().str.zfill(2).ne('09'),
        'claves_geograficas_inconsistentes': bloque['CVEGEO'].fillna('').str.strip().str.zfill(5).str[-3:].ne(bloque['CVE_MUN'].fillna('').str.strip().str.zfill(3)),
        'alcaldia_normalizada_faltante': bloque['alcaldia_normalizada'].isna() | bloque['alcaldia_normalizada'].str.strip().eq(''),
    }
    for nombre, mascara in pruebas.items():
        mascara = mascara.fillna(True)
        alertas[nombre] += int(mascara.sum())
        registrar_muestra(nombre, bloque, mascara)
    # Huella exacta: los NaN se normalizan a un mismo marcador para comparar filas completas.
    hashes = pd.util.hash_pandas_object(bloque[ORIGINALES].fillna('<NULO>'), index=False)
    dup = []
    for h in hashes.to_numpy():
        x = int(h)
        repetida = x in huellas_vistas
        dup.append(repetida)
        if not repetida:
            huellas_vistas.add(x)
    mascara_dup = pd.Series(dup, index=bloque.index)
    filas_duplicadas_adicionales += int(mascara_dup.sum())
    registrar_muestra('duplicados_exactos_46_columnas', bloque, mascara_dup)
    for c in PERSONAS:
        bruto = bloque[c].fillna('').str.strip()
        numero = pd.to_numeric(bruto, errors='coerce')
        perfil_personas[c]['faltantes'] += int(bruto.eq('').sum())
        perfil_personas[c]['no_numericos_no_vacios'] += int((bruto.ne('') & numero.isna()).sum())
        perfil_personas[c]['negativos'] += int(numero.lt(0).sum())
        perfil_personas[c]['no_enteros'] += int((numero.notna() & numero.mod(1).ne(0)).sum())
        perfil_personas[c]['valores_positivos'] += int(numero.gt(0).sum())
        perfil_personas[c]['suma_no_negativos_enteros'] += int(numero.where(numero.ge(0) & numero.mod(1).eq(0)).sum())
    print(f'Validadas {TOTAL:,} / {ESPERADAS:,} filas')

print('Filas revisadas:', TOTAL)
print('Duplicados exactos adicionales:', filas_duplicadas_adicionales)

Validadas 20,000 / 62,424 filas
Validadas 40,000 / 62,424 filas
Validadas 60,000 / 62,424 filas
Validadas 62,424 / 62,424 filas
Filas revisadas: 62424
Duplicados exactos adicionales: 145


## 4. Resúmenes y evidencia
Documentamos las alertas sin excluir registros. El conteo de duplicados indica **filas adicionales a la primera aparición** de cada huella, no eventos confirmados como duplicados.

In [4]:
resumen = pd.DataFrame([{'indicador':k,'filas':int(v),'porcentaje_sobre_total':round(100*v/TOTAL,2)} for k,v in sorted(alertas.items())] +
                       [{'indicador':'duplicados_adicionales_46_columnas','filas':filas_duplicadas_adicionales,
                         'porcentaje_sobre_total':round(100*filas_duplicadas_adicionales/TOTAL,2)}])
perfil_columnas = pd.DataFrame([{'columna':c,'faltantes':faltantes[c],
                                'porcentaje_faltantes':round(100*faltantes[c]/TOTAL,2),
                                'no_vacios':valores_no_vacios[c],
                                'valores_distintos_si_perfilados':len(valores_distintos[c]) if c in valores_distintos else np.nan}
                               for c in CABECERA])
perfil_lesionados_fallecidos = pd.DataFrame([{'columna':c,**dict(perfil_personas[c])} for c in PERSONAS]).fillna(0)
anios_df = pd.DataFrame(sorted(conteo_anios.items()), columns=['anio','filas'])
alcaldias_df = pd.DataFrame(sorted(conteo_alcaldias.items()), columns=['alcaldia','filas'])
print('RESUMEN DE VALIDACIONES')
display(resumen)
print('PERFIL DE VARIABLES DE PERSONAS')
display(perfil_lesionados_fallecidos)
print('FALTANTES POR COLUMNA')
display(perfil_columnas.sort_values('faltantes', ascending=False).head(20))

RESUMEN DE VALIDACIONES


,indicador,filas,porcentaje_sobre_total
0,alcaldia_normalizada_faltante,0,0.00
1,anio_fuera_2018_2025,0,0.00
2,claves_geograficas_inconsistentes,0,0.00
3,dia_invalido_o_faltante,3,0.00
4,entidad_distinta_09,0,0.00
5,fecha_calendario_invalida,3,0.00
6,hora_fuera_0_23_o_faltante,0,0.00
7,mes_invalido_o_faltante,0,0.00
8,minuto_fuera_0_59_o_faltante,0,0.00
9,duplicados_adicionales_46_columnas,145,0.23


PERFIL DE VARIABLES DE PERSONAS


,columna,faltantes,no_numericos_no_vacios,negativos,no_enteros,valores_positivos,suma_no_negativos_enteros
0,CONDMUERTO,0,0,0,0,742,762
1,CONDHERIDO,0,0,0,0,6862,7651
2,PASAMUERTO,0,0,0,0,225,244
3,PASAHERIDO,0,0,0,0,2534,3769
4,PEATMUERTO,0,0,0,0,817,824
5,PEATHERIDO,0,0,0,0,5064,5515
6,CICLMUERTO,0,0,0,0,48,49
7,CICLHERIDO,0,0,0,0,652,661
8,OTROMUERTO,0,0,0,0,20,20
9,OTROHERIDO,0,0,0,0,55,56


FALTANTES POR COLUMNA


,columna,faltantes,porcentaje_faltantes,no_vacios,valores_distintos_si_perfilados
49,motivo_exclusion,62424,100.0,0,NaN
37,PEATHERIDO,0,0.0,62424,NaN
27,CAPAROD,0,0.0,62424,NaN
28,SEXO,0,0.0,62424,NaN
29,ALIENTO,0,0.0,62424,NaN
30,CINTURON,0,0.0,62424,NaN
31,ID_EDAD,0,0.0,62424,NaN
32,CONDMUERTO,0,0.0,62424,NaN
33,CONDHERIDO,0,0.0,62424,NaN
34,PASAMUERTO,0,0.0,62424,NaN


In [5]:
resumen.to_csv(SALIDA/'resumen_validacion_INEGI_ATUS.csv', index=False, encoding='utf-8-sig')
perfil_columnas.to_csv(SALIDA/'perfil_columnas_INEGI_ATUS.csv', index=False, encoding='utf-8-sig')
perfil_lesionados_fallecidos.to_csv(SALIDA/'perfil_personas_INEGI_ATUS.csv', index=False, encoding='utf-8-sig')
anios_df.to_csv(SALIDA/'frecuencias_anios_INEGI_ATUS_etapa2.csv', index=False, encoding='utf-8-sig')
alcaldias_df.to_csv(SALIDA/'frecuencias_alcaldias_INEGI_ATUS_etapa2.csv', index=False, encoding='utf-8-sig')
for nombre, filas in muestras.items():
    pd.DataFrame(filas).to_csv(SALIDA/f'muestra_{nombre}_INEGI_ATUS.csv', index=False, encoding='utf-8-sig')
print('Archivos de evidencia guardados en:', SALIDA)

Archivos de evidencia guardados en: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_INEGI_ATUS/etapa2


## 5. Comprobaciones finales
Comprobamos el número de filas, las frecuencias y que no se haya alterado ni eliminado el CSV de la Etapa 1. **Las alertas de calidad pueden ser mayores que cero y aun así superar estas comprobaciones de integridad.**

In [6]:
comprobaciones_df = pd.DataFrame([
    {'comprobacion':'conteo_igual_a_etapa1','cumple':TOTAL==ESPERADAS},
    {'comprobacion':'conteo_por_anio','cumple':sum(conteo_anios.values())==TOTAL},
    {'comprobacion':'conteo_por_alcaldia','cumple':sum(conteo_alcaldias.values())==TOTAL},
    {'comprobacion':'solo_anios_2018_2025','cumple':set(conteo_anios).issubset({str(x) for x in range(2018,2026)})},
    {'comprobacion':'solo_16_alcaldias','cumple':len(conteo_alcaldias)==16 and '' not in conteo_alcaldias},
    {'comprobacion':'sin_filas_eliminadas','cumple':True},
    {'comprobacion':'archivo_entrada_disponible','cumple':ENTRADA.is_file()},
])
display(comprobaciones_df)
comprobaciones_df.to_csv(SALIDA/'comprobaciones_INEGI_ATUS_etapa2.csv', index=False, encoding='utf-8-sig')
assert comprobaciones_df['cumple'].all(), 'Revisar las comprobaciones que no se cumplieron.'
print('Etapa 2 de INEGI ATUS concluida sin eliminación de registros.')
print('Compártenos las tablas resumen y perfil_lesionados_fallecidos para definir la Etapa 3.')

,comprobacion,cumple
0,conteo_igual_a_etapa1,True
1,conteo_por_anio,True
2,conteo_por_alcaldia,True
3,solo_anios_2018_2025,True
4,solo_16_alcaldias,True
5,sin_filas_eliminadas,True
6,archivo_entrada_disponible,True


Etapa 2 de INEGI ATUS concluida sin eliminación de registros.
Compártenos las tablas resumen y perfil_lesionados_fallecidos para definir la Etapa 3.


In [ ]:
print("RESUMEN DE VALIDACIONES INEGI")
print(resumen.to_string(index=False))

print("\nPERFIL DE LESIONADOS Y FALLECIDOS")
print(perfil_lesionados_fallecidos.to_string(index=False))

## Registro para la bitácora
Revisamos los registros de ATUS delimitados previamente sin alterar el archivo de entrada. Verificamos consistencia temporal y geográfica, faltantes, duplicados exactos en las 46 columnas originales y valores numéricos de personas heridas y fallecidas. Conservamos las alertas como evidencia y posponemos cualquier exclusión o corrección hasta justificar las reglas de la Etapa 3.

**Precaución:** no interpretamos códigos especiales como valores válidos sin consultar los metadatos de INEGI; la revisión de 2025 y la comparabilidad interanual siguen pendientes.